# 01 — Build Corpus and FAISS Index

This notebook builds and evaluates the retrieval infrastructure for the
**Basic RAG** project using HotpotQA.

## Pipeline

HotpotQA  
→ Sample Selection  
→ Corpus Construction  
→ Text Embeddings  
→ FAISS Index  
→ Dense Retrieval  
→ Retrieval Evaluation

## Dataset

HotpotQA is loaded directly from Hugging Face.

For this initial experiment:

- 100 samples are selected from the validation split.
- 10 samples are used for the initial retrieval evaluation.
- Each HotpotQA context paragraph is treated as one retrieval chunk.
- Supporting facts are preserved in the dataset for evaluation only and are
  not provided to the retriever.

## Embedding Model

`sentence-transformers/all-MiniLM-L6-v2`

## Vector Index

FAISS `IndexFlatIP` is used with normalized embeddings.  
Because the embeddings are normalized, inner product corresponds to cosine
similarity.

## Evaluation

The initial retrieval evaluation uses:

- Recall@5
- Retrieval Success Rate

This notebook does not use an LLM and does not perform answer generation.

## Project Configuration

Project artifacts are stored under `Basic_RAG` in Google Drive.

The Google Drive path is used as the runtime storage location in Google Colab.
Generated datasets, corpus files, embeddings, FAISS indexes, and evaluation
results are not intended to be committed to GitHub.

In [3]:
# ============================================================
# Cell 1 — Project Configuration
# ============================================================

# Google Drive project root
PROJECT_ROOT = "/content/drive/MyDrive/Basic_RAG"

# Dataset
DATASET_NAME = "hotpotqa"
DATASET_DIR = f"{PROJECT_ROOT}/data/{DATASET_NAME}"

# Corpus and index
CORPUS_DIR = f"{PROJECT_ROOT}/corpus/{DATASET_NAME}"
INDEX_DIR = f"{PROJECT_ROOT}/indexes/{DATASET_NAME}"

# Evaluation results
RESULTS_DIR = f"{PROJECT_ROOT}/results/{DATASET_NAME}"

# Notebooks
NOTEBOOK_DIR = f"{PROJECT_ROOT}/notebooks"

# Reproducibility
RANDOM_SEED = 42

# Dataset size for this first experiment
N_SAMPLES = 100
N_EVAL = 10

# Retrieval setting used later in Simple RAG
DEFAULT_TOP_K = 5

print("✓ Configuration loaded.")
print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset      : {DATASET_NAME}")
print(f"Samples      : {N_SAMPLES}")
print(f"Evaluation   : {N_EVAL}")
print(f"Top-k        : {DEFAULT_TOP_K}")

✓ Configuration loaded.
Project root : /content/drive/MyDrive/Basic_RAG
Dataset      : hotpotqa
Samples      : 100
Evaluation   : 10
Top-k        : 5


## Dependencies

Install the Python packages required for dataset loading, text embedding,
and FAISS-based vector retrieval.

In [4]:
# ============================================================
# Cell 2 — Install Dependencies
# ============================================================

!pip install -q -U datasets sentence-transformers faiss-cpu

print("✓ Dependencies installed.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 65.8 MB/s eta 0:00:00
✓ Dependencies installed.


## Google Drive and project directories

mount Drive and create the project directories.

In [5]:
# ============================================================
# Cell 3 — Mount Google Drive & Create Project Directories
# ============================================================

from google.colab import drive
import os

# Mount Google Drive
drive.mount("/content/drive")

# Create project directories
directories = [
    PROJECT_ROOT,
    DATASET_DIR,
    CORPUS_DIR,
    INDEX_DIR,
    RESULTS_DIR,
    NOTEBOOK_DIR,
]

for directory in directories:
    os.makedirs(directory, exist_ok=True)

print("✓ Google Drive mounted.")
print("✓ Project directories are ready.\n")

print("Project structure:")
print(f"  {PROJECT_ROOT}/")
print(f"  ├── data/{DATASET_NAME}/")
print(f"  ├── corpus/{DATASET_NAME}/")
print(f"  ├── indexes/{DATASET_NAME}/")
print(f"  ├── results/{DATASET_NAME}/")
print(f"  └── notebooks/")

Mounted at /content/drive
✓ Google Drive mounted.
✓ Project directories are ready.

Project structure:
  /content/drive/MyDrive/Basic_RAG/
  ├── data/hotpotqa/
  ├── corpus/hotpotqa/
  ├── indexes/hotpotqa/
  ├── results/hotpotqa/
  └── notebooks/


## Load HotpotQA

Load the **distractor** version of HotpotQA directly from Hugging Face.

No local copy of the dataset is required.

In [ ]:
# ============================================================
# Cell 4 — Load HotpotQA
# ============================================================

from datasets import load_dataset

# Load the distractor version of HotpotQA
hotpotqa = load_dataset(
    "hotpotqa/hotpot_qa",
    "distractor"
)

print("✓ HotpotQA loaded successfully.\n")

print("Dataset structure:")
print(hotpotqa)

print("\nAvailable splits:")
print(hotpotqa.keys())

✓ HotpotQA loaded successfully.

Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
        num_rows: 90447
    })
    validation: Dataset({
        features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
        num_rows: 7405
    })
})

Available splits:
dict_keys(['train', 'validation'])


## Inspect One HotpotQA Sample

In [ ]:
# ============================================================
# Cell 5 — Inspect HotpotQA Context Structure
# ============================================================

sample = hotpotqa["validation"][0]

print("Sample ID:")
print(sample["id"])

print("\n" + "=" * 80)
print("Question:")
print(sample["question"])

print("\n" + "=" * 80)
print("Gold Answer:")
print(sample["answer"])

print("\n" + "=" * 80)
print("Type:")
print(sample["type"])

print("\n" + "=" * 80)
print("Level:")
print(sample["level"])

print("\n" + "=" * 80)
print("Supporting Facts:")
print(sample["supporting_facts"])

print("\n" + "=" * 80)
print("Context keys:")
print(sample["context"].keys())

print("\n" + "=" * 80)
print("Context structure:")

for key, value in sample["context"].items():
    print(f"\nKey: {key}")
    print(f"Type: {type(value)}")

    if hasattr(value, "__len__"):
        print(f"Length: {len(value)}")

    print("Value preview:")
    print(str(value)[:1000])

Sample ID:
5a8b57f25542995d1e6f1371

Question:
Were Scott Derrickson and Ed Wood of the same nationality?

Gold Answer:
yes

Type:
comparison

Level:
hard

Supporting Facts:
{'title': ['Scott Derrickson', 'Ed Wood'], 'sent_id': [0, 0]}

Context keys:
dict_keys(['title', 'sentences'])

Context structure:

Key: title
Type: <class 'list'>
Length: 10
Value preview:
['Ed Wood (film)', 'Scott Derrickson', 'Woodson, Arkansas', 'Tyler Bates', 'Ed Wood', 'Deliver Us from Evil (2014 film)', 'Adam Collis', 'Sinister (film)', 'Conrad Brooks', 'Doctor Strange (2016 film)']

Key: sentences
Type: <class 'list'>
Length: 10
Value preview:
[['Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.', " The film concerns the period in Wood's life when he made his best-known films as well as his relationship with actor Bela Lugosi, played by Martin Landau.", ' Sarah Jessica Parker, Patricia Arquette, Jeffrey J

## Select 100 samples and 10 evaluation samples

use a fixed random seed and persist the selected sample IDs for reproducibility.

In [ ]:
# ============================================================
# Cell 6 — Select 100 Samples and 10 Evaluation Samples
# ============================================================

import random

# Use the validation split for our development dataset
validation_data = hotpotqa["validation"]

# Create a reproducible list of indices
indices = list(range(len(validation_data)))

random.seed(RANDOM_SEED)
random.shuffle(indices)

# Select 100 samples
selected_indices = indices[:N_SAMPLES]

# Select 10 evaluation samples from the same 100 samples
eval_indices = selected_indices[:N_EVAL]

print(f"Total validation samples : {len(validation_data)}")
print(f"Selected samples         : {len(selected_indices)}")
print(f"Evaluation samples       : {len(eval_indices)}")

print("\nFirst 10 selected sample IDs:")

for i in selected_indices[:10]:
    print(validation_data[i]["id"])

print("\nEvaluation sample IDs:")

for i in eval_indices:
    print(validation_data[i]["id"])


Total validation samples : 7405
Selected samples         : 100
Evaluation samples       : 10

First 10 selected sample IDs:
5a7a567255429941d65f25bd
5abca1a55542993a06baf937
5a73977d554299623ed4ac08
5ab514c05542991779162d72
5add2b435542990d50227e11
5a88d6df554299206df2b377
5ae6b6065542991bbc976168
5ae531ee5542990ba0bbb1ff
5a8aa5835542996c9b8d5f4e
5ab82d095542990e739ec853

Evaluation sample IDs:
5a7a567255429941d65f25bd
5abca1a55542993a06baf937
5a73977d554299623ed4ac08
5ab514c05542991779162d72
5add2b435542990d50227e11
5a88d6df554299206df2b377
5ae6b6065542991bbc976168
5ae531ee5542990ba0bbb1ff
5a8aa5835542996c9b8d5f4e
5ab82d095542990e739ec853


## Save Selected Samples

In [ ]:
# ============================================================
# Cell 7 — Save Selected Samples
# ============================================================

import json

# Convert selected indices to actual dataset records
selected_samples = [
    validation_data[i]
    for i in selected_indices
]

evaluation_samples = [
    validation_data[i]
    for i in eval_indices
]

# Save selected 100 samples
selected_path = f"{DATASET_DIR}/selected_100.json"

with open(selected_path, "w", encoding="utf-8") as f:
    json.dump(
        selected_samples,
        f,
        ensure_ascii=False,
        indent=2
    )

# Save evaluation 10 samples
evaluation_path = f"{DATASET_DIR}/evaluation_10.json"

with open(evaluation_path, "w", encoding="utf-8") as f:
    json.dump(
        evaluation_samples,
        f,
        ensure_ascii=False,
        indent=2
    )

print("✓ Selected samples saved.")
print(f"\n100 samples:")
print(selected_path)

print(f"\n10 evaluation samples:")
print(evaluation_path)

print("\nSaved counts:")
print(f"  Selected   : {len(selected_samples)}")
print(f"  Evaluation : {len(evaluation_samples)}")

✓ Selected samples saved.

100 samples:
/content/drive/MyDrive/Basic_RAG/data/hotpotqa/selected_100.json

10 evaluation samples:
/content/drive/MyDrive/Basic_RAG/data/hotpotqa/evaluation_10.json

Saved counts:
  Selected   : 100
  Evaluation : 10


## Build Chunks from One Sample

In [ ]:
# ============================================================
# Cell 8 — Build Chunks from One Sample
# ============================================================

sample = selected_samples[0]

chunks = []

titles = sample["context"]["title"]
sentence_groups = sample["context"]["sentences"]

for idx, (title, sentences) in enumerate(
    zip(titles, sentence_groups)
):
    # Combine all sentences of one paragraph into one text chunk
    text = " ".join(sentences).strip()

    chunk = {
        "chunk_id": f'{sample["id"]}_chunk_{idx}',
        "sample_id": sample["id"],
        "title": title,
        "text": text,
        "dataset": DATASET_NAME,
        "source_type": "context"
    }

    chunks.append(chunk)

print(f"Sample ID: {sample['id']}")
print(f"Number of chunks: {len(chunks)}")

print("\n" + "=" * 80)

for i, chunk in enumerate(chunks):
    print(f"\nChunk {i}")
    print(f"chunk_id   : {chunk['chunk_id']}")
    print(f"sample_id  : {chunk['sample_id']}")
    print(f"title      : {chunk['title']}")
    print(f"dataset    : {chunk['dataset']}")
    print(f"source_type: {chunk['source_type']}")
    print(f"text       : {chunk['text'][:500]}...")

Sample ID: 5a7a567255429941d65f25bd
Number of chunks: 10


Chunk 0
chunk_id   : 5a7a567255429941d65f25bd_chunk_0
sample_id  : 5a7a567255429941d65f25bd
title      : Radar Station (film)
dataset    : hotpotqa
source_type: context
text       : Radar Station is a 1953 Canadian short documentary film produced by the National Film Board of Canada (NFB) as part of the "On The Spot" series made specifically for television.  The documentary involved an account of a visit to a radar station while it is involved in a simulated air attack, and is based on first-person interviews of the staff at the radar station....

Chunk 1
chunk_id   : 5a7a567255429941d65f25bd_chunk_1
sample_id  : 5a7a567255429941d65f25bd
title      : RAF Trimingham
dataset    : hotpotqa
source_type: context
text       : Remote Radar Head Trimingham or RRH Trimingham is a TPS-77 radar station situated on the coast in the English county of Norfolk.  The site is located on the coast road between Cromer and Mundesley, 1 kilometre e

## Build Full Corpus
represent each HotpotQA context paragraph as one chunk and preserve evaluation-only supporting-fact information separately.

In [ ]:
# ============================================================
# Cell 9 — Build Full Corpus
# ============================================================

import json

corpus = []

for sample in selected_samples:
    sample_id = sample["id"]

    titles = sample["context"]["title"]
    sentence_groups = sample["context"]["sentences"]

    for idx, (title, sentences) in enumerate(
        zip(titles, sentence_groups)
    ):
        # Each paragraph is one chunk
        text = " ".join(sentences).strip()

        chunk = {
            "chunk_id": f"{sample_id}_chunk_{idx}",
            "sample_id": sample_id,
            "title": title,
            "text": text,
            "dataset": DATASET_NAME,
            "source_type": "context"
        }

        corpus.append(chunk)

# Save corpus
corpus_path = f"{CORPUS_DIR}/corpus.json"

with open(corpus_path, "w", encoding="utf-8") as f:
    json.dump(
        corpus,
        f,
        ensure_ascii=False,
        indent=2
    )

print("✓ Corpus built successfully.")
print(f"\nTotal chunks: {len(corpus)}")
print(f"Saved to: {corpus_path}")

# Basic sanity checks
print("\nSanity checks:")
print(f"  Number of samples : {len(selected_samples)}")
print(f"  Number of chunks  : {len(corpus)}")
print(f"  Unique chunk IDs  : {len(set(c['chunk_id'] for c in corpus))}")

print("\nFirst chunk:")
print(json.dumps(corpus[0], ensure_ascii=False, indent=2))


✓ Corpus built successfully.

Total chunks: 995
Saved to: /content/drive/MyDrive/Basic_RAG/corpus/hotpotqa/corpus.json

Sanity checks:
  Number of samples : 100
  Number of chunks  : 995
  Unique chunk IDs  : 995

First chunk:
{
  "chunk_id": "5a7a567255429941d65f25bd_chunk_0",
  "sample_id": "5a7a567255429941d65f25bd",
  "title": "Radar Station (film)",
  "text": "Radar Station is a 1953 Canadian short documentary film produced by the National Film Board of Canada (NFB) as part of the \"On The Spot\" series made specifically for television.  The documentary involved an account of a visit to a radar station while it is involved in a simulated air attack, and is based on first-person interviews of the staff at the radar station.",
  "dataset": "hotpotqa",
  "source_type": "context"
}


## Corpus Sanity Check

In [ ]:
# ============================================================
# Cell 10 — Corpus Sanity Check
# ============================================================

import statistics

# Check empty fields
empty_chunk_ids = [
    c["chunk_id"]
    for c in corpus
    if not c["chunk_id"]
]

empty_titles = [
    c["chunk_id"]
    for c in corpus
    if not c["title"]
]

empty_texts = [
    c["chunk_id"]
    for c in corpus
    if not c["text"].strip()
]

# Calculate text lengths
text_lengths = [
    len(c["text"])
    for c in corpus
]

print("Corpus sanity check")
print("=" * 60)

print(f"Total chunks       : {len(corpus)}")
print(f"Unique chunk IDs   : {len(set(c['chunk_id'] for c in corpus))}")

print(f"\nEmpty chunk IDs    : {len(empty_chunk_ids)}")
print(f"Empty titles       : {len(empty_titles)}")
print(f"Empty texts        : {len(empty_texts)}")

print("\nText length statistics:")
print(f"  Minimum          : {min(text_lengths)} characters")
print(f"  Maximum          : {max(text_lengths)} characters")
print(f"  Mean             : {statistics.mean(text_lengths):.1f} characters")
print(f"  Median           : {statistics.median(text_lengths):.1f} characters")

print("\nFirst 3 chunk titles:")
for chunk in corpus[:3]:
    print(f"  - {chunk['title']}")

Corpus sanity check
Total chunks       : 995
Unique chunk IDs   : 995

Empty chunk IDs    : 0
Empty titles       : 0
Empty texts        : 0

Text length statistics:
  Minimum          : 52 characters
  Maximum          : 2734 characters
  Mean             : 555.6 characters
  Median           : 491.0 characters

First 3 chunk titles:
  - Radar Station (film)
  - RAF Trimingham
  - NATO Ouvrage &quot;G&quot;


## Load embedding model

Model: `sentence-transformers/all-MiniLM-L6-v2`

In [ ]:
# ============================================================
# Cell 11 — Load Embedding Model
# ============================================================

from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print("✓ Embedding model loaded.")
print(f"Model: {EMBEDDING_MODEL_NAME}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✓ Embedding model loaded.
Model: sentence-transformers/all-MiniLM-L6-v2


## Create normalized embeddings

encode corpus texts and normalize embeddings.

In [ ]:
# ============================================================
# Cell 12 — Generate Normalized Embeddings
# ============================================================

import numpy as np

# Extract chunk texts
corpus_texts = [
    chunk["text"]
    for chunk in corpus
]

# Generate normalized embeddings
embeddings = embedding_model.encode(
    corpus_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

# Convert to NumPy array
embeddings = np.asarray(embeddings, dtype="float32")

print("\n✓ Embeddings generated.")

print(f"Number of embeddings : {embeddings.shape[0]}")
print(f"Embedding dimension  : {embeddings.shape[1]}")
print(f"Embedding dtype      : {embeddings.dtype}")
print(f"Embedding shape      : {embeddings.shape}")


Batches:   0%|          | 0/32 [00:00<?, ?it/s]


✓ Embeddings generated.
Number of embeddings : 995
Embedding dimension  : 384
Embedding dtype      : float32
Embedding shape      : (995, 384)


## Save Embeddings

In [ ]:
# ============================================================
# Cell 13 — Save Embeddings
# ============================================================

import numpy as np
import os

embeddings_path = f"{INDEX_DIR}/embeddings.npy"

np.save(
    embeddings_path,
    embeddings
)

print("✓ Embeddings saved.")
print(f"Path: {embeddings_path}")

# Verify the saved file
saved_embeddings = np.load(embeddings_path)

print("\nVerification:")
print(f"  Shape : {saved_embeddings.shape}")
print(f"  Dtype : {saved_embeddings.dtype}")

✓ Embeddings saved.
Path: /content/drive/MyDrive/Basic_RAG/indexes/hotpotqa/embeddings.npy

Verification:
  Shape : (995, 384)
  Dtype : float32


## Build FAISS Index

use `faiss.IndexFlatIP` over normalized embeddings.

In [ ]:
# ============================================================
# Cell 14 — Build FAISS Index
# ============================================================

import faiss

# Embedding dimension
embedding_dim = embeddings.shape[1]

# Create Inner Product index
index = faiss.IndexFlatIP(embedding_dim)

# Add normalized embeddings to the index
index.add(embeddings)

print("✓ FAISS index built.")

print(f"Index type       : {type(index).__name__}")
print(f"Embedding dim    : {embedding_dim}")
print(f"Number of vectors: {index.ntotal}")


✓ FAISS index built.
Index type       : IndexFlatIP
Embedding dim    : 384
Number of vectors: 995


## Save FAISS Index


In [ ]:
# ============================================================
# Cell 15 — Save FAISS Index
# ============================================================

import faiss

index_path = f"{INDEX_DIR}/faiss.index"

faiss.write_index(
    index,
    index_path
)

print("✓ FAISS index saved.")
print(f"Path: {index_path}")

# Verify the saved index
loaded_index = faiss.read_index(index_path)

print("\nVerification:")
print(f"  Index type        : {type(loaded_index).__name__}")
print(f"  Number of vectors : {loaded_index.ntotal}")
print(f"  Embedding dim     : {loaded_index.d}")


✓ FAISS index saved.
Path: /content/drive/MyDrive/Basic_RAG/indexes/hotpotqa/faiss.index

Verification:
  Index type        : IndexFlatIP
  Number of vectors : 995
  Embedding dim     : 384


## Retrieval smoke test

A small retrieval test to verify that the saved index can be queried.

In [ ]:
# ============================================================
# Cell 16 — Retrieval Smoke Test
# ============================================================

# Select one evaluation question
test_sample = evaluation_samples[0]

query = test_sample["question"]

print("Query:")
print(query)

print("\n" + "=" * 80)

# Encode the query
query_embedding = embedding_model.encode(
    [query],
    normalize_embeddings=True
)

query_embedding = np.asarray(
    query_embedding,
    dtype="float32"
)

# Retrieve top-k chunks
scores, indices = index.search(
    query_embedding,
    DEFAULT_TOP_K
)

print(f"Top-{DEFAULT_TOP_K} retrieved chunks:\n")

for rank, (score, idx) in enumerate(
    zip(scores[0], indices[0]),
    start=1
):
    chunk = corpus[idx]

    print(f"Rank {rank}")
    print(f"Score    : {score:.4f}")
    print(f"Chunk ID : {chunk['chunk_id']}")
    print(f"Title    : {chunk['title']}")
    print(f"Text     : {chunk['text'][:300]}...")
    print("-" * 80)


Query:
What was Iqbal F. Qadir on when he participated in an attack on a radar station located on western shore of the Okhamandal Peninsula?

Top-5 retrieved chunks:

Rank 1
Score    : 0.5720
Chunk ID : 5a7a567255429941d65f25bd_chunk_9
Title    : Iqbal F. Qadir
Text     : Vice-Admiral Iqbal Fazl Quadir (Urdu:اقبال فضل قادر) , is a retired three-star rank admiral in the Pakistan Navy, former diplomat, and a defence analyst.  He is renown for his participation in second war with India when he was part of the flotilla that attacked the radar station in Dwarka, India....
--------------------------------------------------------------------------------
Rank 2
Score    : 0.5118
Chunk ID : 5a7a567255429941d65f25bd_chunk_4
Title    : Mukachevo Radar Station
Text     : Mukachevo radar station was a Soviet radar station providing early warning of ballistic missile attack.  It was located in Shipka in the far south west of Ukraine and was part of the Soviet, and then Russian missile attack warning

## Check Retrieved Chunks Against Supporting Facts

Compare the retrieved chunks with the gold supporting facts for one evaluation
sample.

Supporting facts are used here **only for evaluation** and are not provided
to the retriever.

In [ ]:
# ============================================================
# Cell 17 — Check Retrieved Chunks Against Supporting Facts
# ============================================================

# Gold supporting facts
supporting_facts = test_sample["supporting_facts"]

gold_titles = set(
    supporting_facts["title"]
)

print("Gold supporting fact titles:")
for title in gold_titles:
    print(f"  - {title}")

print("\n" + "=" * 80)

# Retrieved chunks
retrieved_chunks = [
    corpus[idx]
    for idx in indices[0]
]

print("Retrieved titles:")
for rank, chunk in enumerate(retrieved_chunks, start=1):
    is_gold = chunk["title"] in gold_titles

    print(
        f"Rank {rank}: "
        f"{chunk['title']} "
        f"--> {'✓ GOLD EVIDENCE' if is_gold else 'not gold evidence'}"
    )

Gold supporting fact titles:
  - Dwarka
  - Iqbal F. Qadir

Retrieved titles:
Rank 1: Iqbal F. Qadir --> ✓ GOLD EVIDENCE
Rank 2: Mukachevo Radar Station --> not gold evidence
Rank 3: Sevastopol Radar Station --> not gold evidence
Rank 4: Radar Station B-71 --> not gold evidence
Rank 5: No. 227 Radar Station RAAF --> not gold evidence


## Reusable FAISS Retriever

Question → Embedding → FAISS → Top-k Documents

In [ ]:
# ============================================================
# Cell 18 — Reusable FAISS Retriever
# ============================================================

class FAISSRetriever:
    """
    Simple reusable retriever based on:
    SentenceTransformer + FAISS IndexFlatIP
    """

    def __init__(self, embedding_model, index, corpus):
        self.embedding_model = embedding_model
        self.index = index
        self.corpus = corpus

    def retrieve(self, query, top_k=5):
        """
        Retrieve the top-k most similar chunks for a query.
        """

        # Encode query
        query_embedding = self.embedding_model.encode(
            [query],
            normalize_embeddings=True
        )

        query_embedding = np.asarray(
            query_embedding,
            dtype="float32"
        )

        # Search FAISS
        scores, indices = self.index.search(
            query_embedding,
            top_k
        )

        # Build readable results
        results = []

        for rank, (score, idx) in enumerate(
            zip(scores[0], indices[0]),
            start=1
        ):
            chunk = self.corpus[idx]

            result = {
                "rank": rank,
                "score": float(score),
                "chunk_id": chunk["chunk_id"],
                "sample_id": chunk["sample_id"],
                "title": chunk["title"],
                "text": chunk["text"],
                "dataset": chunk["dataset"],
                "source_type": chunk["source_type"]
            }

            results.append(result)

        return results


# Create retriever
retriever = FAISSRetriever(
    embedding_model=embedding_model,
    index=index,
    corpus=corpus
)

print("✓ Retriever created successfully.")


✓ Retriever created successfully.


## Test Reusable Retriever

In [ ]:
# ============================================================
# Cell 19 — Test Reusable Retriever
# ============================================================

test_sample = evaluation_samples[0]

query = test_sample["question"]

results = retriever.retrieve(
    query,
    top_k=DEFAULT_TOP_K
)

print("Query:")
print(query)

print("\n" + "=" * 80)

print(f"Top-{DEFAULT_TOP_K} Retrieved Chunks:\n")

for result in results:
    print(f"Rank {result['rank']}")
    print(f"Score    : {result['score']:.4f}")
    print(f"Chunk ID : {result['chunk_id']}")
    print(f"Sample ID: {result['sample_id']}")
    print(f"Title    : {result['title']}")
    print(f"Text     : {result['text'][:300]}...")
    print("-" * 80)

Query:
What was Iqbal F. Qadir on when he participated in an attack on a radar station located on western shore of the Okhamandal Peninsula?

Top-5 Retrieved Chunks:

Rank 1
Score    : 0.5720
Chunk ID : 5a7a567255429941d65f25bd_chunk_9
Sample ID: 5a7a567255429941d65f25bd
Title    : Iqbal F. Qadir
Text     : Vice-Admiral Iqbal Fazl Quadir (Urdu:اقبال فضل قادر) , is a retired three-star rank admiral in the Pakistan Navy, former diplomat, and a defence analyst.  He is renown for his participation in second war with India when he was part of the flotilla that attacked the radar station in Dwarka, India....
--------------------------------------------------------------------------------
Rank 2
Score    : 0.5118
Chunk ID : 5a7a567255429941d65f25bd_chunk_4
Sample ID: 5a7a567255429941d65f25bd
Title    : Mukachevo Radar Station
Text     : Mukachevo radar station was a Soviet radar station providing early warning of ballistic missile attack.  It was located in Shipka in the far south west of Ukr

## Retrieval Evaluation Function

In [ ]:
# ============================================================
# Cell 20 — Retrieval Evaluation Function
# ============================================================

def evaluate_retrieval(sample, retriever, top_k=5):
    """
    Evaluate retrieval recall for one HotpotQA sample.
    """

    # Retrieve documents
    results = retriever.retrieve(
        sample["question"],
        top_k=top_k
    )

    # Build gold evidence identifiers
    gold_evidence = {
        (sample["id"], title)
        for title in sample["supporting_facts"]["title"]
    }

    # Build retrieved identifiers
    retrieved_evidence = {
        (result["sample_id"], result["title"])
        for result in results
    }

    # Find retrieved gold evidence
    matched_evidence = (
        gold_evidence.intersection(retrieved_evidence)
    )

    # Calculate recall
    recall = (
        len(matched_evidence) / len(gold_evidence)
        if gold_evidence
        else 0.0
    )

    return {
        "sample_id": sample["id"],
        "question": sample["question"],
        "gold_evidence_count": len(gold_evidence),
        "retrieved_gold_count": len(matched_evidence),
        "recall": recall,
        "retrieval_success": len(matched_evidence) > 0,
        "results": results
    }


print("✓ Retrieval evaluation function created.")

✓ Retrieval evaluation function created.


In [ ]:
# ============================================================
# Cell 21 — Evaluate Retrieval on 10 Questions
# ============================================================

retrieval_results = []

for sample in evaluation_samples:

    result = evaluate_retrieval(
        sample,
        retriever,
        top_k=DEFAULT_TOP_K
    )

    retrieval_results.append(result)


print("✓ Retrieval evaluation completed.")
print(f"Number of evaluated questions: {len(retrieval_results)}")

print("\n" + "=" * 100)

for i, result in enumerate(retrieval_results, start=1):

    print(f"\nQuestion {i}")
    print(f"Sample ID             : {result['sample_id']}")
    print(f"Question              : {result['question']}")
    print(f"Gold evidence count   : {result['gold_evidence_count']}")
    print(f"Retrieved gold count  : {result['retrieved_gold_count']}")
    print(f"Recall@{DEFAULT_TOP_K}            : {result['recall']:.2f}")
    print(f"Retrieval success     : {result['retrieval_success']}")

✓ Retrieval evaluation completed.
Number of evaluated questions: 10


Question 1
Sample ID             : 5a7a567255429941d65f25bd
Question              : What was Iqbal F. Qadir on when he participated in an attack on a radar station located on western shore of the Okhamandal Peninsula?
Gold evidence count   : 2
Retrieved gold count  : 1
Recall@5            : 0.50
Retrieval success     : True

Question 2
Sample ID             : 5abca1a55542993a06baf937
Question              : When did the park at which Tivolis Koncertsal is located open?
Gold evidence count   : 2
Retrieved gold count  : 2
Recall@5            : 1.00
Retrieval success     : True

Question 3
Sample ID             : 5a73977d554299623ed4ac08
Question              : What is the shared country of ancestry between Art Laboe and Scout Tufankjian?
Gold evidence count   : 2
Retrieved gold count  : 2
Recall@5            : 1.00
Retrieval success     : True

Question 4
Sample ID             : 5ab514c05542991779162d72
Question       

## Retrieval Summary Metrics

In [ ]:
# ============================================================
# Cell 22 — Retrieval Summary Metrics
# ============================================================

recalls = [
    result["recall"]
    for result in retrieval_results
]

successful_retrievals = [
    result["retrieval_success"]
    for result in retrieval_results
]

mean_recall = sum(recalls) / len(recalls)

retrieval_success_rate = (
    sum(successful_retrievals)
    / len(successful_retrievals)
)

print("Retrieval Evaluation Summary")
print("=" * 60)

print(f"Number of questions : {len(retrieval_results)}")
print(f"Top-k               : {DEFAULT_TOP_K}")

print(f"\nMean Recall@{DEFAULT_TOP_K}       : {mean_recall:.4f}")
print(f"Retrieval Success Rate : {retrieval_success_rate:.4f}")

print("\nRecall distribution:")

for recall_value in sorted(set(recalls), reverse=True):
    count = recalls.count(recall_value)

    print(
        f"  Recall@{DEFAULT_TOP_K} = "
        f"{recall_value:.2f} : {count} questions"
    )

Retrieval Evaluation Summary
Number of questions : 10
Top-k               : 5

Mean Recall@5       : 0.8000
Retrieval Success Rate : 1.0000

Recall distribution:
  Recall@5 = 1.00 : 6 questions
  Recall@5 = 0.50 : 4 questions


## Save Retrieval Evaluation Results

In [ ]:
# ============================================================
# Cell 23 — Save Retrieval Evaluation Results
# ============================================================

import json
import os

retrieval_results_path = (
    f"{RESULTS_DIR}/retrieval_baseline_top{DEFAULT_TOP_K}.json"
)

# Save results
with open(
    retrieval_results_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        retrieval_results,
        f,
        ensure_ascii=False,
        indent=2
    )

print("✓ Retrieval results saved.")
print(f"\nPath:")
print(retrieval_results_path)

print("\nSaved information includes:")
print("  - Question")
print("  - Sample ID")
print("  - Gold evidence count")
print("  - Retrieved gold count")
print(f"  - Recall@{DEFAULT_TOP_K}")
print("  - Retrieval success")
print("  - Retrieved chunks")

✓ Retrieval results saved.

Path:
/content/drive/MyDrive/Basic_RAG/results/hotpotqa/retrieval_baseline_top5.json

Saved information includes:
  - Question
  - Sample ID
  - Gold evidence count
  - Retrieved gold count
  - Recall@5
  - Retrieval success
  - Retrieved chunks
